# Post-Hoc Evaluation — Quality Validator + Qwen-as-Judge

**No pipeline re-runs.** Loads saved JSON configs from `master_run_results/configs/`,
runs structural quality checks locally, then optionally calls Qwen-as-judge.

| Stage | What | Network? | Crashes if missing? |
|---|---|---|---|
| Quality validator | Schema + render checks | No | No |
| Qwen strict | 8 binary checks, PASS if all pass | vLLM 11877 | No — skipped gracefully |
| Qwen lenient | 4 major checks only | vLLM 11877 | No — skipped gracefully |
| Qwen configurable | 8 dimensions scored 0–1 | vLLM 11877 | No — skipped gracefully |

**plt.show() fix applied** — renders save correctly to PNG without blank images.

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup (crash-safe)
# ═══════════════════════════════════════════════════════════════════════
import os, sys, json, csv, copy, time, glob, traceback, re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from openai import OpenAI

# ── Project root
CANDIDATES = [
    '/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch',
    os.path.abspath('..'), os.getcwd(),
]
ROOT = next((p for p in CANDIDATES if os.path.isfile(os.path.join(p, 'generate_visualization.py'))), None)
assert ROOT, 'Cannot find project root — update path list'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from retrieve_data import retrieve_data
from visualization_from_template import generate_from_template

# ── Paths
CONFIGS_DIR = os.path.join(ROOT, 'master_run_results', 'configs')
META_CSV    = os.path.join(ROOT, 'master_run_results', 'master_pipeline_runs.csv')
OUT         = os.path.join(ROOT, 'post_hoc_eval')
RENDERS     = os.path.join(OUT, 'renders')
FIGURES     = os.path.join(OUT, 'figures')
for d in [OUT, RENDERS, FIGURES]:
    os.makedirs(d, exist_ok=True)

# ── Check what exists
configs_exist  = os.path.isdir(CONFIGS_DIR) and len(glob.glob(os.path.join(CONFIGS_DIR, '*.json'))) > 0
meta_exists    = os.path.isfile(META_CSV)
print(f'Configs dir  : {CONFIGS_DIR}')
print(f'  → exists   : {configs_exist} ({len(glob.glob(os.path.join(CONFIGS_DIR,"*.json"))) if configs_exist else 0} JSONs)')
print(f'Meta CSV     : {META_CSV} → exists: {meta_exists}')
print(f'Output dir   : {OUT}')

if not configs_exist:
    print('WARNING: No config JSONs found. Run master_run_pipeline.ipynb first.')
    print(f'  Expected location: {CONFIGS_DIR}')

# ── Data + question
MD_TABLE = retrieve_data(None, type='test')
QUESTION = (
    'Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? '
    'Provide a detailed analysis of the data based of total earnings and '
    'provide a comprehensive visualization supporting your analysis.'
)
print(f'Data words   : {len(MD_TABLE.split())}')

# ── Qwen vLLM — optional, graceful fallback
VLLM_URL = 'http://hal9000.skim.th-owl.de:11877/v1'
JUDGE_THRESHOLD = 0.70
SAMPLE_PER_SCENARIO = None   # None = all, integer = N runs per scenario

try:
    qwen_raw = OpenAI(base_url=VLLM_URL, api_key='dummy', timeout=30)
    available = [m.id for m in qwen_raw.models.list().data]
    QWEN_PRIORITY = ['Qwen3-30B','Qwen2.5-72B','Qwen2.5-32B','Qwen2.5-14B','Qwen']
    QWEN_MODEL = next(
        (m for p in QWEN_PRIORITY for m in available if p.lower() in m.lower()),
        available[0] if available else None
    )
    QWEN_OK = QWEN_MODEL is not None
    if QWEN_OK:
        print(f'vLLM connected. Using model: {QWEN_MODEL}')
    else:
        print('WARNING: vLLM reachable but no models found. Judge will be skipped.')
except Exception as e:
    QWEN_MODEL = 'OFFLINE'
    QWEN_OK    = False
    print(f'WARNING: vLLM not reachable ({e})')
    print('Quality checks will run. Qwen judge cells will be skipped gracefully.')


Configs dir  : /Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/master_run_results/configs
  → exists   : True (180 JSONs)
Meta CSV     : /Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/master_run_results/master_pipeline_runs.csv → exists: True
Output dir   : /Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/post_hoc_eval
Data words   : 345
vLLM connected. Using model: Qwen3.6-27B


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Load metadata CSV + all saved JSON configs
# ═══════════════════════════════════════════════════════════════════════

# Load metadata from master_pipeline_runs.csv if it exists
meta_rows = {}
if meta_exists:
    meta_df = pd.read_csv(META_CSV)
    for _, row in meta_df.iterrows():
        key = (str(row['scenario']).strip(), int(row['run']))
        meta_rows[key] = row.to_dict()
    print(f'Loaded {len(meta_rows)} rows from master_pipeline_runs.csv')
    print(f'Scenarios found: {sorted(meta_df["scenario"].unique())}')
else:
    print('No master_pipeline_runs.csv found — will work from config files only')

# Load all config JSON files
CONFIG_RECORDS = []
json_files = sorted(glob.glob(os.path.join(CONFIGS_DIR, '*.json')))
print(f'\nLoading {len(json_files)} config files...')

for fpath in json_files:
    fname = os.path.basename(fpath)
    # Parse scenario + run from filename: SCENARIO_run01.json
    try:
        parts = fname.replace('.json', '').rsplit('_run', 1)
        scenario = parts[0]
        run_n    = int(parts[1]) if len(parts) > 1 else 0
    except Exception:
        scenario = fname.replace('.json', '')
        run_n    = 0

    try:
        with open(fpath, encoding='utf-8') as f:
            cfg = json.load(f)
        # Skip error-only configs
        if '_error' in cfg and len(cfg) == 1:
            cfg = None
    except Exception:
        cfg = None

    # Pull latency from metadata if available
    meta = meta_rows.get((scenario, run_n), {})
    CONFIG_RECORDS.append({
        'scenario'     : scenario,
        'run'          : run_n,
        'config'       : cfg,
        'config_path'  : fpath,
        'latency_s'    : meta.get('latency_s', None),
        'scenario_desc': meta.get('scenario_desc', scenario),
    })

# Apply sampling if requested
if SAMPLE_PER_SCENARIO:
    from collections import defaultdict
    by_s = defaultdict(list)
    for r in CONFIG_RECORDS:
        by_s[r['scenario']].append(r)
    CONFIG_RECORDS = [r for recs in by_s.values() for r in recs[:SAMPLE_PER_SCENARIO]]
    print(f'Sampled to {SAMPLE_PER_SCENARIO} runs per scenario: {len(CONFIG_RECORDS)} total')

valid_configs = sum(1 for r in CONFIG_RECORDS if r['config'] is not None)
print(f'\nTotal records  : {len(CONFIG_RECORDS)}')
print(f'Valid configs  : {valid_configs}')
print(f'Error/missing  : {len(CONFIG_RECORDS) - valid_configs}')


Loaded 180 rows from master_pipeline_runs.csv
Scenarios found: ['S0', 'S1', 'S2', 'S3', 'S4', 'S4b', 'S5', 'S5b', 'S6', 'S7', 'S8', 'S9', 'SA1', 'SA2', 'SA3', 'SA4', 'SA5', 'SA6']

Loading 180 config files...

Total records  : 180
Valid configs  : 180
Error/missing  : 0


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — Quality validator (local, zero LLM calls)
# ═══════════════════════════════════════════════════════════════════════

VALID_CHARTTYPES = {'line','bar','pie','scatter','column','stackedbar'}

def validate_quality(cfg, scenario='?', run=0, render_tag=None):
    r = dict(
        scenario=scenario, run=run, got_json=(cfg is not None),
        schema_complete=False, valid_charttype=False, has_data=False,
        has_title=False, has_axes=False, renderable=False,
        has_annotations=False, data_consistency=False,
        quality_score=0, passed=False,
        n_data_groups=0, n_annotations=0, charttype='', title='', render_error=''
    )
    if cfg is None:
        return r

    required = {'titlename','charttype','xlabel','ylabel','data'}
    r['schema_complete'] = required.issubset(set(cfg.keys()))

    ct = str(cfg.get('charttype','')).strip().lower()
    if hasattr(cfg.get('charttype'), 'value'):
        ct = cfg['charttype'].value
    r['valid_charttype'] = ct in VALID_CHARTTYPES
    r['charttype'] = ct

    data = cfg.get('data') or []
    if not isinstance(data, list): data = []
    r['has_data']      = len(data) > 0
    r['n_data_groups'] = len(data)
    r['has_title']     = bool(str(cfg.get('titlename','')).strip())
    r['title']         = str(cfg.get('titlename',''))[:60]
    r['has_axes']      = bool(cfg.get('xlabel','')) and bool(cfg.get('ylabel',''))

    annos = cfg.get('annotations') or []
    r['has_annotations'] = len(annos) > 0
    r['n_annotations']   = len(annos)

    ok = True
    for g in data:
        d  = g if isinstance(g, dict) else (g.model_dump() if hasattr(g,'model_dump') else {})
        xd = d.get('x_data') or []; yd = d.get('y_data') or []
        if not xd or not yd or len(xd) != len(yd):
            ok = False; break
    r['data_consistency'] = ok

    # Render check — plt.show() patched so it doesn't clear the figure
    if r['has_data'] and r['valid_charttype']:
        try:
            rc = copy.deepcopy(cfg)
            if hasattr(rc.get('charttype'), 'value'):
                rc['charttype'] = rc['charttype'].value
            # Fix tick mismatches
            xt = rc.get('x_ticks',[]); xl = rc.get('x_tick_label',[])
            if len(xl) != len(xt):
                n = min(len(xt),len(xl)); rc['x_ticks']=xt[:n]; rc['x_tick_label']=xl[:n]
            yt = rc.get('y_ticks',[]); yl = rc.get('y_tick_label',[])
            if len(yl) != len(yt):
                n = min(len(yt),len(yl)); rc['y_ticks']=yt[:n]; rc['y_tick_label']=yl[:n]
            nd = []
            for g in (rc.get('data') or []):
                d  = g if isinstance(g,dict) else g.model_dump()
                xd = d.get('x_data',[]); yd=[float(v) for v in (d.get('y_data') or [])]
                if not isinstance(xd,list): xd=list(range(len(yd)))
                n  = min(len(xd),len(yd))
                nd.append({'label':d.get('label',''),'x_data':xd[:n],'y_data':yd[:n]})
            rc['data'] = nd; rc.setdefault('annotations',[])
            plt.close('all')
            _show = plt.show
            plt.show = lambda *a, **k: None   # KEY FIX: suppress plt.show()
            generate_from_template(rc)
            if render_tag:
                out_path = os.path.join(RENDERS, f'{render_tag}.png')
                plt.savefig(out_path, dpi=100, bbox_inches='tight')
            plt.show = _show
            plt.close('all')
            r['renderable'] = True
        except Exception as e:
            r['render_error'] = str(e)[:100]
            plt.close('all')
            try: plt.show = _show
            except: pass

    SCORED = ['schema_complete','valid_charttype','has_data','has_title',
              'has_axes','renderable','has_annotations','data_consistency']
    r['quality_score'] = sum(r[c] for c in SCORED)
    r['passed']        = r['quality_score'] >= 5
    return r

print('validate_quality() ready — 8-point scorer, PASS = score >= 5/8')
print('plt.show() patched inside renderer — PNGs will save correctly')


validate_quality() ready — 8-point scorer, PASS = score >= 5/8
plt.show() patched inside renderer — PNGs will save correctly


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — Qwen-as-judge prompts + call function
# Skipped gracefully if Qwen offline
# ═══════════════════════════════════════════════════════════════════════

JUDGE_STRICT = (
    'You are a STRICT chart configuration validator. Reject configs with ANY issue.\n\n'
    'Evaluate against 8 rules and respond with valid JSON ONLY (no markdown, no preamble):\n'
    '1. CHART_TYPE_MATCH: charttype matches user intent\n'
    '2. TITLE_QUALITY: title is specific and descriptive\n'
    '3. DATA_COMPLETENESS: all required years/segments present\n'
    '4. DATA_GROUPING: data is grouped logically\n'
    '5. AXIS_LABELS: xlabel/ylabel are clear and correct\n'
    '6. ANNOTATIONS_PRESENT: at least one annotation\n'
    '7. SCALE_REASONABLE: y-axis range is sensible for the data\n'
    '8. NO_HALLUCINATION: data values match the provided table\n\n'
    'Respond ONLY with this JSON:\n'
    '{"verdict": "PASS" or "FAIL", '
    '"checks": {"chart_type_match":true/false,"title_quality":true/false,'
    '"data_completeness":true/false,"data_grouping":true/false,'
    '"axis_labels":true/false,"annotations_present":true/false,'
    '"scale_reasonable":true/false,"no_hallucination":true/false}, '
    '"issues": ["..."], "fix_suggestion": "..."}'
)

JUDGE_LENIENT = (
    'You are a LENIENT chart validator. Only FAIL for serious errors.\n\n'
    'Check 4 things (JSON only, no preamble):\n'
    '1. wrong_data: true if values are clearly hallucinated or wrong\n'
    '2. empty_output: true if chart has no data groups\n'
    '3. severely_wrong_chart: true if chart type makes no sense\n'
    '4. hallucination: true if title/labels reference data not in the table\n\n'
    'Respond ONLY with:\n'
    '{"verdict": "PASS" or "FAIL", '
    '"checks": {"wrong_data":false,"empty_output":false,'
    '"severely_wrong_chart":false,"hallucination":false}, '
    '"issues": []}'
)

JUDGE_CONFIGURABLE = (
    f'You are a chart quality evaluator. Score 8 dimensions 0.0–1.0.\n'
    f'PASS threshold: mean score >= {JUDGE_THRESHOLD}\n\n'
    'Rate each dimension 0.0 (bad) to 1.0 (perfect). JSON only, no preamble:\n'
    '{"dimensions": {'
    '"chart_type_match":0.0-1.0, "title_specificity":0.0-1.0, '
    '"data_completeness":0.0-1.0, "data_grouping_quality":0.0-1.0, '
    '"axis_label_quality":0.0-1.0, "annotation_quality":0.0-1.0, '
    '"scale_reasonableness":0.0-1.0, "data_fidelity":0.0-1.0}, '
    '"mean_score":0.0-1.0, '
    '"verdict": "PASS" or "FAIL"}'
)

JUDGE_PROMPTS = {'strict': JUDGE_STRICT, 'lenient': JUDGE_LENIENT, 'configurable': JUDGE_CONFIGURABLE}

def run_qwen_judge(question, md_table, cfg, mode):
    '''Call Qwen-as-judge. Returns (parsed_dict, latency_s, error_msg).'''
    if not QWEN_OK:
        return None, 0.0, 'vLLM offline'
    prompt = JUDGE_PROMPTS.get(mode, JUDGE_STRICT)
    user_content = (
        f'User query: {question}\n\n'
        f'Data table (first 200 chars): {md_table[:200]}...\n\n'
        f'Chart config to evaluate:\n{json.dumps(cfg, default=str, indent=2)[:2000]}'
    )
    t0 = time.perf_counter()
    try:
        resp = qwen_raw.chat.completions.create(
            model=QWEN_MODEL,
            messages=[{'role':'system','content':prompt}, {'role':'user','content':user_content}],
            temperature=0, max_tokens=400, timeout=60,
        )
        lat = round(time.perf_counter()-t0, 3)
        text = resp.choices[0].message.content.strip()
        # Strip markdown fences if present
        text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.MULTILINE)
        text = re.sub(r'\s*```$', '', text, flags=re.MULTILINE).strip()
        parsed = json.loads(text)
        return parsed, lat, None
    except json.JSONDecodeError as e:
        return None, round(time.perf_counter()-t0,3), f'JSON parse error: {e}'
    except Exception as e:
        return None, round(time.perf_counter()-t0,3), str(e)[:100]

print(f'Qwen judge ready. Online={QWEN_OK}  Model={QWEN_MODEL}')
print('Judge will be skipped gracefully if offline — quality checks always run.')


Qwen judge ready. Online=True  Model=Qwen3.6-27B
Judge will be skipped gracefully if offline — quality checks always run.


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Run quality checks on all configs (no LLM, fast)
# ═══════════════════════════════════════════════════════════════════════

QUALITY_CSV = os.path.join(OUT, 'quality.csv')
Q_FIELDS = [
    'scenario','run','latency_s','scenario_desc',
    'got_json','schema_complete','valid_charttype','has_data',
    'has_title','has_axes','renderable','has_annotations','data_consistency',
    'n_data_groups','n_annotations','charttype','title',
    'quality_score','passed','render_error'
]

quality_results = []
print(f'Running quality checks on {len(CONFIG_RECORDS)} configs...\n')

with open(QUALITY_CSV, 'w', newline='', encoding='utf-8') as fout:
    writer = csv.DictWriter(fout, fieldnames=Q_FIELDS, extrasaction='ignore')
    writer.writeheader()

    for i, rec in enumerate(CONFIG_RECORDS, 1):
        scen = rec['scenario']; run = rec['run']; cfg = rec['config']
        tag  = f'{scen}_run{run:02d}'
        qr   = validate_quality(cfg, scenario=scen, run=run, render_tag=tag)
        row  = {
            'scenario'     : scen,
            'run'          : run,
            'latency_s'    : rec.get('latency_s',''),
            'scenario_desc': rec.get('scenario_desc', scen),
            **{k: qr.get(k) for k in Q_FIELDS if k not in ('scenario','run','latency_s','scenario_desc')}
        }
        quality_results.append(row)
        with open(QUALITY_CSV, 'a', newline='', encoding='utf-8') as fa:
            csv.DictWriter(fa, fieldnames=Q_FIELDS, extrasaction='ignore').writerow(row)
        sym = 'PASS' if qr['passed'] else 'FAIL'
        rnd = 'render:OK' if qr['renderable'] else f'render:FAIL({qr["render_error"][:30]})'
        print(f'  {i:>3}/{len(CONFIG_RECORDS)}  {tag:<20}  {sym}  Q={qr["quality_score"]}/8  {rnd}')

print(f'\n Saved -> {QUALITY_CSV}')
n_pass = sum(1 for r in quality_results if r['passed'])
n_rend = sum(1 for r in quality_results if r['renderable'])
print(f'Pass rate  : {n_pass}/{len(quality_results)} ({100*n_pass/max(len(quality_results),1):.1f}%)')
print(f'Render rate: {n_rend}/{len(quality_results)} ({100*n_rend/max(len(quality_results),1):.1f}%)')


Running quality checks on 180 configs...

    1/180  S0_run01              PASS  Q=8/8  render:OK
    2/180  S0_run02              PASS  Q=8/8  render:OK
    3/180  S0_run03              PASS  Q=8/8  render:OK
    4/180  S0_run04              PASS  Q=8/8  render:OK
    5/180  S0_run05              PASS  Q=8/8  render:OK
    6/180  S0_run06              PASS  Q=8/8  render:OK
    7/180  S0_run07              PASS  Q=8/8  render:OK
    8/180  S0_run08              PASS  Q=8/8  render:OK
    9/180  S0_run09              PASS  Q=8/8  render:OK
   10/180  S0_run10              PASS  Q=8/8  render:OK
   11/180  S1_run01              PASS  Q=8/8  render:OK
   12/180  S1_run02              PASS  Q=8/8  render:OK
   13/180  S1_run03              PASS  Q=8/8  render:OK
   14/180  S1_run04              PASS  Q=8/8  render:OK
   15/180  S1_run05              PASS  Q=8/8  render:OK
   16/180  S1_run06              PASS  Q=8/8  render:OK
   17/180  S1_run07              PASS  Q=8/8  render:OK
   18/

/var/folders/zy/jrpsl1v90xx47xwd7fws6np00000gn/T/ipykernel_60064/2106407115.py:75: UserWarning: Glyph 8208 (\N{HYPHEN}) missing from font(s) Arial.
  plt.savefig(out_path, dpi=100, bbox_inches='tight')


   19/180  S1_run09              PASS  Q=8/8  render:OK
   20/180  S1_run10              PASS  Q=8/8  render:OK
   21/180  S2_run01              PASS  Q=8/8  render:OK
   22/180  S2_run02              PASS  Q=8/8  render:OK
   23/180  S2_run03              PASS  Q=8/8  render:OK
   24/180  S2_run04              PASS  Q=8/8  render:OK
   25/180  S2_run05              PASS  Q=8/8  render:OK
   26/180  S2_run06              PASS  Q=8/8  render:OK
   27/180  S2_run07              PASS  Q=8/8  render:OK
   28/180  S2_run08              PASS  Q=8/8  render:OK
   29/180  S2_run09              PASS  Q=8/8  render:OK
   30/180  S2_run10              PASS  Q=8/8  render:OK
   31/180  S3_run01              PASS  Q=8/8  render:OK
   32/180  S3_run02              PASS  Q=8/8  render:OK
   33/180  S3_run03              PASS  Q=8/8  render:OK
   34/180  S3_run04              PASS  Q=8/8  render:OK
   35/180  S3_run05              PASS  Q=8/8  render:OK
   36/180  S3_run06              PASS  Q=8/8  re

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Qwen-as-judge loop
# Crash-safe: writes row-by-row. Skipped entirely if Qwen offline.
# ═══════════════════════════════════════════════════════════════════════

JUDGE_CSV = os.path.join(OUT, 'judge.csv')
J_FIELDS = [
    'scenario','run','mode','judge_latency_s','verdict','score','issues','fix_suggestion',
    's_chart_type_match','s_title_quality','s_data_completeness','s_data_grouping',
    's_axis_labels','s_annotations_present','s_scale_reasonable','s_no_hallucination',
    'l_wrong_data','l_empty_output','l_severely_wrong_chart','l_hallucination',
    'c_chart_type_match','c_title_specificity','c_data_completeness',
    'c_data_grouping_quality','c_axis_label_quality','c_annotation_quality',
    'c_scale_reasonableness','c_data_fidelity','c_mean_score','error',
]

judgeable = [r for r in CONFIG_RECORDS if r['config'] is not None]

if not QWEN_OK:
    print('Qwen is offline — skipping judge loop.')
    print('Quality checks (Cell 5) are complete. Figures will use quality.csv only.')
    # Write empty judge CSV so Cell 7 does not crash
    with open(JUDGE_CSV, 'w', newline='', encoding='utf-8') as fout:
        csv.DictWriter(fout, fieldnames=J_FIELDS).writeheader()
    print(f'Empty judge CSV written -> {JUDGE_CSV}')
else:
    MODES = ['strict', 'lenient', 'configurable']
    n_judge = len(judgeable)
    print(f'Qwen judge: {n_judge} configs x {len(MODES)} modes = {n_judge*len(MODES)} calls')
    print(f'Model     : {QWEN_MODEL}')

    judge_results = []
    t_start = time.perf_counter()

    with open(JUDGE_CSV, 'w', newline='', encoding='utf-8') as fout:
        writer = csv.DictWriter(fout, fieldnames=J_FIELDS, extrasaction='ignore')
        writer.writeheader()

    for idx, rec in enumerate(judgeable, 1):
        scen = rec['scenario']; run = rec['run']; cfg = rec['config']
        elapsed = time.perf_counter()-t_start
        eta     = (elapsed/idx)*(n_judge-idx) if idx > 1 else 0
        print(f'  [{idx:>3}/{n_judge}] {scen}_run{run:02d}  ETA {eta/60:.1f}min', end='')

        for mode in MODES:
            verdict, j_lat, err = run_qwen_judge(QUESTION, MD_TABLE, cfg, mode)
            row = {
                'scenario':scen, 'run':run, 'mode':mode,
                'judge_latency_s': j_lat,
                'verdict' : verdict.get('verdict','ERROR') if verdict else 'ERROR',
                'score'   : verdict.get('score') or verdict.get('mean_score','') if verdict else '',
                'issues'  : ' | '.join(verdict.get('issues',[]) or []) if verdict else '',
                'fix_suggestion': (verdict.get('fix_suggestion') or '') if verdict else '',
                'error'   : err or '',
            }
            if verdict and mode == 'strict':
                ch = verdict.get('checks',{}) or {}
                row.update({
                    's_chart_type_match'  : ch.get('chart_type_match',''),
                    's_title_quality'     : ch.get('title_quality',''),
                    's_data_completeness' : ch.get('data_completeness',''),
                    's_data_grouping'     : ch.get('data_grouping',''),
                    's_axis_labels'       : ch.get('axis_labels',''),
                    's_annotations_present': ch.get('annotations_present',''),
                    's_scale_reasonable'  : ch.get('scale_reasonable',''),
                    's_no_hallucination'  : ch.get('no_hallucination',''),
                })
            if verdict and mode == 'lenient':
                ch = verdict.get('checks',{}) or {}
                row.update({
                    'l_wrong_data'           : ch.get('wrong_data',''),
                    'l_empty_output'         : ch.get('empty_output',''),
                    'l_severely_wrong_chart' : ch.get('severely_wrong_chart',''),
                    'l_hallucination'        : ch.get('hallucination',''),
                })
            if verdict and mode == 'configurable':
                dims = verdict.get('dimensions',{}) or {}
                row.update({
                    'c_chart_type_match'     : dims.get('chart_type_match',''),
                    'c_title_specificity'    : dims.get('title_specificity',''),
                    'c_data_completeness'    : dims.get('data_completeness',''),
                    'c_data_grouping_quality': dims.get('data_grouping_quality',''),
                    'c_axis_label_quality'   : dims.get('axis_label_quality',''),
                    'c_annotation_quality'   : dims.get('annotation_quality',''),
                    'c_scale_reasonableness' : dims.get('scale_reasonableness',''),
                    'c_data_fidelity'        : dims.get('data_fidelity',''),
                    'c_mean_score'           : verdict.get('mean_score',''),
                })
            judge_results.append(row)
            with open(JUDGE_CSV, 'a', newline='', encoding='utf-8') as fa:
                csv.DictWriter(fa, fieldnames=J_FIELDS, extrasaction='ignore').writerow(row)
            sym = 'V' if row['verdict']=='PASS' else ('X' if row['verdict']=='FAIL' else '?')
            print(f'  {mode[0].upper()}{sym}', end='')
        print()

    total_t = time.perf_counter()-t_start
    print(f'\nDone. {total_t/60:.1f} min total.')
    print(f'Saved -> {JUDGE_CSV}')


Qwen judge: 180 configs x 3 modes = 540 calls
Model     : Qwen3.6-27B
  [  1/180] S0_run01  ETA 0.0min  S?  L?  C?
  [  2/180] S0_run02  ETA 96.7min  S?  L?  C?
  [  3/180] S0_run03  ETA 94.6min  S?  L?  C?
  [  4/180] S0_run04  ETA 114.3min  S?  L?  C?
  [  5/180] S0_run05  ETA 131.3min  S?  L?  C?
  [  6/180] S0_run06  ETA 149.1min  S?  L?  C?
  [  7/180] S0_run07  ETA 150.3min  S?  L?  C?
  [  8/180] S0_run08  ETA 152.4min  S?  L?  C?
  [  9/180] S0_run09  ETA 153.0min  S?  L?  C?
  [ 10/180] S0_run10  ETA 151.0min  S?  L?  C?
  [ 11/180] S1_run01  ETA 151.5min  S?  L?  C?
  [ 12/180] S1_run02  ETA 151.8min  S?  L?  C?
  [ 13/180] S1_run03  ETA 152.2min  S?  L?  C?
  [ 14/180] S1_run04  ETA 150.1min  S?  L?  C?
  [ 15/180] S1_run05  ETA 144.3min  S?  L?  C?
  [ 16/180] S1_run06  ETA 144.2min  S?  L?  C?
  [ 17/180] S1_run07  ETA 144.4min  S?  L?  C?
  [ 18/180] S1_run08  ETA 144.0min  S?  L?  C?
  [ 19/180] S1_run09  ETA 147.3min  S?  L?  C?
  [ 20/180] S1_run10  ETA 146.6min  S?  L

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Combine quality + judge → combined.csv + summary.csv
# Works even if judge.csv is empty (Qwen offline)
# ═══════════════════════════════════════════════════════════════════════

q_df = pd.read_csv(QUALITY_CSV)
COMBINED_CSV = os.path.join(OUT, 'combined.csv')
SUMMARY_CSV  = os.path.join(OUT, 'summary.csv')

# Load judge results — graceful if empty or missing
HAS_JUDGE = False
if os.path.isfile(JUDGE_CSV):
    j_df = pd.read_csv(JUDGE_CSV)
    if len(j_df) > 0 and 'scenario' in j_df.columns:
        HAS_JUDGE = True
        print(f'Judge CSV loaded: {len(j_df)} rows')
    else:
        print('Judge CSV exists but is empty (Qwen was offline) — proceeding with quality data only')
else:
    print('No judge CSV found — proceeding with quality data only')

# Merge if judge data available
if HAS_JUDGE:
    try:
        piv = j_df.pivot_table(
            index=['scenario','run'],
            columns='mode',
            values=['verdict','score'],
            aggfunc='first'
        )
        piv.columns = [f'{col[1]}_{col[0]}' for col in piv.columns]
        piv = piv.reset_index()
        combined = q_df.merge(piv, on=['scenario','run'], how='left')
        print(f'Merged quality + judge: {len(combined)} rows')
    except Exception as e:
        print(f'Merge failed ({e}) — using quality only')
        combined = q_df.copy(); HAS_JUDGE = False
else:
    combined = q_df.copy()

combined.to_csv(COMBINED_CSV, index=False)
print(f'Saved -> {COMBINED_CSV}')

# Build summary by scenario
CALL_MAP = {
    'S0':4,'S1':3,'S2':2,'S3':1,'S4':3,'S4b':3,'S5':2,'S5b':2,'S6':1,'S7':3,'S8':2,'S9':1,
    'SA1':4,'SA2':4,'SA3':4,'SA4':3,'SA5':2,'SA6':3,
}

agg = {
    'latency_s'    : ['mean','median','std','min','max'],
    'quality_score': 'mean',
    'passed'       : 'mean',
    'renderable'   : 'mean',
    'n_data_groups': 'mean',
    'n_annotations': 'mean',
}
if HAS_JUDGE:
    for col in ['strict_verdict','lenient_verdict','configurable_verdict']:
        if col in combined.columns:
            agg[col] = lambda x: (x=='PASS').mean()

summary_df = combined.groupby(['scenario','scenario_desc']).agg(agg)
summary_df.columns = ['_'.join(c).strip('_') for c in summary_df.columns]
summary_df = summary_df.reset_index()
summary_df['calls'] = summary_df['scenario'].map(lambda s: str(CALL_MAP.get(s,'?'))+'-call')
summary_df['pass_pct']   = (summary_df['passed_mean'] * 100).round(1)
summary_df['render_pct'] = (summary_df['renderable_mean'] * 100).round(1)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f'Saved -> {SUMMARY_CSV}')

# Print summary
print(f"\n{'='*80}")
print('SUMMARY BY SCENARIO')
print(f"{'='*80}")
cols_to_show = ['scenario','calls','latency_s_mean','latency_s_median','pass_pct','render_pct','quality_score_mean']
cols_to_show = [c for c in cols_to_show if c in summary_df.columns]
print(summary_df[cols_to_show].to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Figures (works with or without judge data)
# ═══════════════════════════════════════════════════════════════════════

smry = summary_df.copy()
CALL_COLORS = {'1-call':'#1D9E75','2-call':'#378ADD','3-call':'#E67E22','4-call':'#C0392B'}
smry['color'] = smry['calls'].map(CALL_COLORS).fillna('#888')
x = np.arange(len(smry))
labels = smry['scenario'].tolist()

# ── Fig 1: pass rate vs render rate ─────────────────────────────────
fig, ax = plt.subplots(figsize=(16,5))
bw = 0.3
ax.bar(x - bw/2, smry['pass_pct'],   bw, color=smry['color'], alpha=0.9, label='Pass rate (%)')
ax.bar(x + bw/2, smry['render_pct'], bw, color=smry['color'], alpha=0.5, hatch='//', label='Render rate (%)')
ax.axhline(80, color='gray', ls='--', lw=1.2, alpha=0.6, label='80% target')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Rate (%)'); ax.set_ylim(0, 115)
ax.set_title('Structural Quality: Pass Rate vs Render Success — re-evaluated from saved JSONs',
             fontweight='bold')
patches = [Patch(color=v, label=k) for k,v in CALL_COLORS.items()]
ax.legend(handles=patches + ax.get_legend_handles_labels()[0][:2], fontsize=8, loc='lower right')
ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig1_path = os.path.join(FIGURES, 'fig1_pass_render.png')
plt.savefig(fig1_path, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved -> {fig1_path}')

# ── Fig 2: mean quality score ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16,5))
bars = ax.bar(x, smry['quality_score_mean'], color=smry['color'], alpha=0.9)
for bar, v in zip(bars, smry['quality_score_mean']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{v:.1f}',
            ha='center', fontsize=8, fontweight='bold')
ax.axhline(5, color='gray', ls='--', lw=1.2, alpha=0.6, label='Pass threshold (5/8)')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean quality score (max 8)'); ax.set_ylim(0, 9)
ax.set_title('Mean Structural Quality Score per Scenario', fontweight='bold')
patches = [Patch(color=v, label=k) for k,v in CALL_COLORS.items()]
ax.legend(handles=patches, fontsize=8, loc='lower right')
ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig2_path = os.path.join(FIGURES, 'fig2_quality.png')
plt.savefig(fig2_path, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved -> {fig2_path}')

# ── Fig 3: latency distribution (box plots) ──────────────────────────
fig, ax = plt.subplots(figsize=(16,5))
lats_by_scen = []
for scen in labels:
    lats = combined[combined['scenario']==scen]['latency_s'].dropna().tolist()
    lats_by_scen.append(lats)
bp = ax.boxplot(lats_by_scen, positions=x, widths=0.6, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], smry['color']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Latency (seconds)')
ax.set_title('Latency Distribution per Scenario (N=10 runs each)', fontweight='bold')
ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig3_path = os.path.join(FIGURES, 'fig3_latency.png')
plt.savefig(fig3_path, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved -> {fig3_path}')

# ── Fig 4: Pareto — quality vs latency ──────────────────────────────
fig, ax = plt.subplots(figsize=(12,7))
for _, row in smry.iterrows():
    qscore = row.get('quality_score_mean',0)
    lat    = row.get('latency_s_mean', row.get('latency_s_mean',0))
    calls  = row.get('calls','?')
    color  = CALL_COLORS.get(calls,'#888')
    ax.scatter(lat, qscore/8*100, color=color, s=100, zorder=3)
    ax.annotate(row['scenario'], (lat, qscore/8*100), fontsize=8,
                xytext=(3,3), textcoords='offset points')
ax.set_xlabel('Mean Latency (seconds)')
ax.set_ylabel('Mean Quality Score (% of max)')
ax.set_title('Quality vs Latency — Pareto view (ideal = top-left)', fontweight='bold', fontsize=12)
patches = [Patch(color=v, label=k) for k,v in CALL_COLORS.items()]
ax.legend(handles=patches, fontsize=9)
ax.spines[['top','right']].set_visible(False); ax.grid(alpha=0.3)
plt.tight_layout()
fig4_path = os.path.join(FIGURES, 'fig4_pareto.png')
plt.savefig(fig4_path, dpi=150, bbox_inches='tight'); plt.close()
print(f'Saved -> {fig4_path}')

# ── Fig 5: Judge results (only if Qwen ran) ──────────────────────────
if HAS_JUDGE:
    MODES = ['strict','lenient','configurable']
    verdict_cols = [c for c in summary_df.columns if 'verdict' in c]
    if verdict_cols:
        fig, ax = plt.subplots(figsize=(16,5))
        bw2 = 0.25
        mcolors = ['#1D9E75','#378ADD','#E67E22']
        for mi, (mode, mc) in enumerate(zip(MODES, mcolors)):
            col = f'{mode}_verdict_<lambda_0>'
            if col not in smry.columns:
                col = next((c for c in smry.columns if mode in c and 'verdict' in c), None)
            if col:
                vals = smry[col].fillna(0) * 100
                ax.bar(x + (mi-1)*bw2, vals, bw2, label=f'Qwen {mode}', color=mc, alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('PASS rate (%)'); ax.set_ylim(0,115)
        ax.set_title('Qwen-as-Judge PASS Rate by Mode and Scenario', fontweight='bold')
        ax.legend(fontsize=9)
        ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        fig5_path = os.path.join(FIGURES, 'fig5_judge.png')
        plt.savefig(fig5_path, dpi=150, bbox_inches='tight'); plt.close()
        print(f'Saved -> {fig5_path}')
else:
    print('Fig 5 (judge) skipped — Qwen was offline')

print(f'\nAll figures saved to: {FIGURES}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Display rendered charts inline
# Shows one sample render per scenario for visual inspection
# ═══════════════════════════════════════════════════════════════════════

from IPython.display import display, Image
import glob

render_files = sorted(glob.glob(os.path.join(RENDERS, '*.png')))
print(f'Total renders: {len(render_files)}')

# Show one render per scenario (run 1 preferred)
shown = set()
for scenario in smry['scenario'].tolist():
    matches = [f for f in render_files if os.path.basename(f).startswith(scenario + '_run01')]
    if not matches:
        matches = [f for f in render_files if os.path.basename(f).startswith(scenario)]
    if matches and scenario not in shown:
        shown.add(scenario)
        fpath = matches[0]
        desc  = smry[smry['scenario']==scenario]['scenario_desc'].values
        label = desc[0][:60] if len(desc) > 0 else scenario
        print(f'\n  {scenario}: {label}')
        try:
            display(Image(fpath, width=580))
        except Exception as e:
            print(f'  (display error: {e})')
